In [0]:
readings_raw = spark.read.csv('/Volumes/workspace/sensorpulse/sensorpulse_data/readings_raw.csv', header=True, inferSchema=True)
display(readings_raw)


reading_id,sensor_id,value,unit,recorded_at,is_anomaly
2d9ec247-3358-499b-89b6-1a27f522bf50,25f7fe15-ac58-46fd-a2a5-8ac722e51b07,35.82,°C,2025-03-30T04:54:08.333Z,false
15dfdcb8-034f-497c-ae09-8272c5937032,8c8f4128-73fb-439c-8494-2e9368f359ac,82.61,AQI,2025-04-17T23:26:12.410Z,false
0c0bba6f-23fd-4400-9235-8e0d1ae3123f,41b8a75d-2f1f-4899-a578-038192551466,16.19,°C,2025-10-29T06:10:49.672Z,false
397538d2-eaed-4f17-a54a-8f3f5760f286,6f8a94a4-16ab-4662-8a6b-92bec646d246,150.39,AQI,2024-11-16T09:54:17.454Z,false
a2965942-5fa9-41bd-b82f-8288eaa92270,b473dbc3-424e-4c82-b04d-5aea82aee8d1,62.4,%,2025-03-12T02:20:29.617Z,false
8abf477d-ecf2-4697-8ccc-5ae8c3ceef5b,6a0e671e-ecfa-4948-b70c-e74b2c42bc11,54.66,%,2025-06-29T09:43:22.582Z,false
74c4dc68-89de-4cb8-99d3-32b9d7130c2b,311b7165-d3ba-4bbf-9c55-56a78e0b4d83,55.62,%,2025-06-01T12:12:14.352Z,false
dced2416-42b7-424b-bbae-cb09b4baa2e6,646a0529-f908-47ff-9a2b-24f82cdb6663,106.19,AQI,2026-03-15T05:23:58.947Z,false
307b93c6-604b-47ac-b138-d5c322b0ea3a,dfc68d11-d658-4fbd-b75d-5fd45664df54,65.34,%,2026-06-11T11:07:42.297Z,false
08275ac7-dfbc-4f06-927c-c9fff208d06b,027a0b86-ece9-40ee-b7ca-688336495999,64.38,%,2024-12-04T09:12:47.038Z,false


In [0]:
from pyspark.sql.functions import current_timestamp, lit

readings = readings_raw.withColumn('ingested_at', current_timestamp()).withColumn('ingest_source', lit('readings_raw.csv'))

readings.write.mode('delta').mode('overwrite').saveAsTable("workspace.sensorpulse.batch_readings")


In [0]:
readings.write.mode('delta').mode('overwrite').saveAsTable("workspace.sensorpulse.batch_readings")

In [0]:
sensors_raw = spark.read.csv('/Volumes/workspace/sensorpulse/sensorpulse_data/sensors_raw.csv', header=True, inferSchema=True)

In [0]:
display(sensors_raw)

sensor_id,district,sensor_type,latitude_longitude,installed_at
25f7fe15-ac58-46fd-a2a5-8ac722e51b07,Gloucestershire,temperature,"56.395715, -133.325072",2023-08-16
8c8f4128-73fb-439c-8494-2e9368f359ac,Scottish Borders,air_quality,"-31.3098155, -55.091133",2024-02-12
41b8a75d-2f1f-4899-a578-038192551466,Gloucestershire,temperature,"84.4520245, 112.563230",2024-11-05
6f8a94a4-16ab-4662-8a6b-92bec646d246,Hertfordshire,air_quality,"-15.323751, -176.511005",2024-10-08
b473dbc3-424e-4c82-b04d-5aea82aee8d1,Tyne and Wear,humidity,"-15.4094795, -96.527578",2023-11-01
6a0e671e-ecfa-4948-b70c-e74b2c42bc11,Greater London,humidity,"11.9859595, -128.075790",2025-04-19
311b7165-d3ba-4bbf-9c55-56a78e0b4d83,Oxfordshire,humidity,"-78.336362, 66.651852",2023-10-08
646a0529-f908-47ff-9a2b-24f82cdb6663,Greater London,air_quality,"-11.300556, 157.498421",2024-02-05
dfc68d11-d658-4fbd-b75d-5fd45664df54,Tyne and Wear,humidity,"-71.328931, -155.398221",2026-05-31
027a0b86-ece9-40ee-b7ca-688336495999,Gloucestershire,humidity,"-27.5106725, -125.775270",2024-11-10


In [0]:
%sql

DROP TABLE IF EXISTS workspace.sensorpulse.batch_sensors;

In [0]:
from pyspark.sql.functions import to_date
sensors = sensors_raw \
    .withColumn('installed_at', to_date('installed_at')) \
    .withColumn('ingested_at', current_timestamp()) \
    .withColumn('ingest_source', lit('sensors_raw.csv'))
display(sensors)
sensors.write.mode('delta').mode('overwrite').saveAsTable("workspace.sensorpulse.batch_sensors")

sensor_id,district,sensor_type,latitude_longitude,installed_at,ingested_at,ingest_source
25f7fe15-ac58-46fd-a2a5-8ac722e51b07,Gloucestershire,temperature,"56.395715, -133.325072",2023-08-16,2026-07-16T11:38:31.691Z,sensors_raw.csv
8c8f4128-73fb-439c-8494-2e9368f359ac,Scottish Borders,air_quality,"-31.3098155, -55.091133",2024-02-12,2026-07-16T11:38:31.691Z,sensors_raw.csv
41b8a75d-2f1f-4899-a578-038192551466,Gloucestershire,temperature,"84.4520245, 112.563230",2024-11-05,2026-07-16T11:38:31.691Z,sensors_raw.csv
6f8a94a4-16ab-4662-8a6b-92bec646d246,Hertfordshire,air_quality,"-15.323751, -176.511005",2024-10-08,2026-07-16T11:38:31.691Z,sensors_raw.csv
b473dbc3-424e-4c82-b04d-5aea82aee8d1,Tyne and Wear,humidity,"-15.4094795, -96.527578",2023-11-01,2026-07-16T11:38:31.691Z,sensors_raw.csv
6a0e671e-ecfa-4948-b70c-e74b2c42bc11,Greater London,humidity,"11.9859595, -128.075790",2025-04-19,2026-07-16T11:38:31.691Z,sensors_raw.csv
311b7165-d3ba-4bbf-9c55-56a78e0b4d83,Oxfordshire,humidity,"-78.336362, 66.651852",2023-10-08,2026-07-16T11:38:31.691Z,sensors_raw.csv
646a0529-f908-47ff-9a2b-24f82cdb6663,Greater London,air_quality,"-11.300556, 157.498421",2024-02-05,2026-07-16T11:38:31.691Z,sensors_raw.csv
dfc68d11-d658-4fbd-b75d-5fd45664df54,Tyne and Wear,humidity,"-71.328931, -155.398221",2026-05-31,2026-07-16T11:38:31.691Z,sensors_raw.csv
027a0b86-ece9-40ee-b7ca-688336495999,Gloucestershire,humidity,"-27.5106725, -125.775270",2024-11-10,2026-07-16T11:38:31.691Z,sensors_raw.csv


In [0]:
from pyspark.sql import Window

In [0]:
%sql
DESCRIBE workspace.sensorpulse.batch_readings;

col_name,data_type,comment
reading_id,string,null
sensor_id,string,null
value,double,null
unit,string,null
recorded_at,timestamp,null
is_anomaly,boolean,null
ingested_at,timestamp,null
ingest_source,string,null


In [0]:
from pyspark.sql.functions import avg, to_date, col, round

daily_aggregates = readings \
    .withColumn("reading_date", to_date(col("recorded_at"))) \
    .join(sensors.select("sensor_id", "district", "sensor_type"), "sensor_id") \
    .groupBy("sensor_type", "district", "reading_date") \
    .agg(
        round(avg("value"), 2).alias("avg_value")
    )

daily_aggregates.write.mode('delta').mode('overwrite').saveAsTable("workspace.sensorpulse.daily_aggregates")
display(daily_aggregates)

sensor_type,district,reading_date,avg_value
humidity,Gloucestershire,2024-12-04,64.38
temperature,Tyne and Wear,2024-10-20,28.01
temperature,Tyne and Wear,2024-06-24,10.54
humidity,Tyne and Wear,2024-09-06,35.2
air_quality,Scottish Borders,2026-06-14,101.64
air_quality,South Yorkshire,2025-10-17,144.99
humidity,Gloucestershire,2026-05-11,43.19
temperature,Gloucestershire,2026-05-16,14.94
air_quality,Gloucestershire,2026-07-02,94.64
air_quality,Scottish Borders,2025-10-14,93.15


In [0]:
from pyspark.sql.functions import count, sum as spark_sum

anomaly_stats = readings_raw \
    .withColumn("reading_date", to_date(col("recorded_at"))) \
    .join(sensors_raw.select("sensor_id", "district", 'sensor_type'), "sensor_id") \
    .groupBy("sensor_type", "district", "reading_date") \
    .agg(
        count("reading_id").alias("total_readings"),
        spark_sum(col("is_anomaly").cast("int")).alias("anomaly_count"),
        round(
            spark_sum(col("is_anomaly").cast("int")) / count("reading_id") * 100, 2
        ).alias("anomaly_rate_pct")
    )

anomaly_stats.write.format("delta").mode("overwrite").saveAsTable("workspace.sensorpulse.batch_anomaly_stats")

In [0]:
display(anomaly_stats)

sensor_type,district,reading_date,total_readings,anomaly_count,anomaly_rate_pct
humidity,Gloucestershire,2024-12-04,1,0,0.0
temperature,Tyne and Wear,2024-10-20,1,0,0.0
temperature,Tyne and Wear,2024-06-24,1,0,0.0
humidity,Tyne and Wear,2024-09-06,1,0,0.0
air_quality,Scottish Borders,2026-06-14,1,0,0.0
air_quality,South Yorkshire,2025-10-17,1,0,0.0
humidity,Gloucestershire,2026-05-11,1,0,0.0
temperature,Gloucestershire,2026-05-16,1,0,0.0
air_quality,Gloucestershire,2026-07-02,7,0,0.0
air_quality,Scottish Borders,2025-10-14,1,0,0.0
